In [48]:
import pandas as pd
import numpy as np
import os

In [49]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/ig-filter-data")

print("Path to dataset files:", path)

100%|██████████| 7.13M/7.13M [00:00<00:00, 48.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/ig-filter-data/versions/2


In [50]:
df = pd.read_csv(path + "/ganjoo_ipl_bbb.csv")

Converting BBB to MBM

In [92]:
def process_batting_scorecard_from_kaggle(df):
    """Process Kaggle ball-by-ball data into a batting scorecard format."""

    # Dictionary to store batting statistics. Key = (match_id, batting_team, striker, innings)
    batting_stats = {}

    # Process match by match
    for match_id in df['p_match'].unique():
        match_df = df[df['p_match'] == match_id]

        # Get the match date for this specific match
        match_date = match_df['date'].iloc[0] if 'date' in match_df.columns and not pd.isna(match_df['date'].iloc[0]) else ""

        for _, ball in match_df.iterrows():
            try:
                striker = pd.to_numeric(ball['p_bat'], errors='coerce')
                batting_team = ball['team_bat']
                innings = ball['inns']

                # Create unique key for each batsman in a match
                key = (match_id, batting_team, striker, innings)

                # Initialize stats for new batsman
                if key not in batting_stats:
                    batting_stats[key] = {
                        'batsman_name': ball['bat'],
                        'runs': 0,
                        'balls': 0,
                        'fours': 0,
                        'sixes': 0,
                        'isOut': False,
                        'wicketType': 'not out',
                        'bowler_name': '',
                        'p_bowl_id': None,
                        'match_date': match_date  # Store the date with each batsman's record
                    }

                stats = batting_stats[key]

                # Count legal deliveries - using noball and wide columns directly
                is_wide = pd.notna(ball['wide']) and ball['wide'] > 0
                is_noball = pd.notna(ball['noball']) and ball['noball'] > 0

                if not (is_wide or is_noball):
                    stats['balls'] += 1

                # Add runs scored by batsman
                runs = ball['batruns'] if pd.notna(ball['batruns']) else 0
                stats['runs'] += runs

                # Count boundaries
                if runs == 4:
                    stats['fours'] += 1
                elif runs == 6:
                    stats['sixes'] += 1

                # Check for dismissal
                out_status = str(ball['bat_out']).strip().upper() if 'bat_out' in ball and pd.notna(ball['bat_out']) else ""
                if out_status == "" and pd.notna(ball['out']) and str(ball['out']).strip().upper() == 'TRUE':
                    out_status = 'TRUE'

                dismissed_player = str(ball['p_bat']).strip() if pd.notna(ball['p_bat']) else ""

                if out_status == 'TRUE' and dismissed_player == str(striker):
                    stats['isOut'] = True
                    stats['wicketType'] = str(ball['dismissal']).strip() if pd.notna(ball['dismissal']) else "unknown"
                    stats['bowler_name'] = ball['bowl'] if pd.notna(ball['bowl']) else ""
                    stats['p_bowl_id'] = pd.to_numeric(ball['p_bowl'], errors='coerce') if pd.notna(ball['p_bowl']) else None

                batting_stats[key] = stats
            except Exception as e:
                print(f"Error processing row in match {match_id}: {e}")
                continue

    results = []
    for (match_id, team, player, innings), stats in batting_stats.items():
        strike_rate = (stats['runs'] / stats['balls'] * 100) if stats['balls'] > 0 else 0
        results.append({
            'MatchID': match_id,
            'date': stats['match_date'],  # Use the stored date for each batsman
            'innings': innings,
            'team': team,
            'batsman_id': player,
            'batsman_name': stats['batsman_name'],
            'runs': stats['runs'],
            'balls': stats['balls'],
            'fours': stats['fours'],
            'sixes': stats['sixes'],
            'strikeRate': round(strike_rate, 2),
            'wicketType': stats['wicketType'],
            'bowler_name': stats['bowler_name'] if stats['isOut'] else '',
        })

    return pd.DataFrame(results)

In [93]:
new_df = process_batting_scorecard_from_kaggle(df)

In [102]:
new_df.to_csv("ganjoo_ipl_mbm_bat.csv", index=False)